# Compression Data Cleaning Tool

Use this notebook before the rate-dependent pad/PCB stack calculator when instrument resolution produces repeated values on either the displacement or force channel while the other channel continues to change.

The notebook preserves the original file, cleans each loading curve independently, compares the raw and cleaned data, and exports a calculator-ready CSV. It supports:

- **PAD** data with columns `Rate_mm_min, Disp_mm, Force_N`
- **PCB** data with columns `Disp_mm, Force_N`

> Keep the raw test file as the permanent measurement record. Always review the comparison plot before using the cleaned file.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

## Customer inputs

Set `DATA_TYPE` to `"PAD"` or `"PCB"`, then set `RESOLUTION_AXIS` to `"DISPLACEMENT"` or `"FORCE"` according to the channel producing repeated readings. Enter the applicable instrument resolution. A median-filter window of 5 is a conservative starting point; use 1 to disable filtering.

On Windows, a complete path can be entered as `Path("D:/folder/file.csv")` or `Path(r"D:\\folder\\file.csv")`.

In [ ]:
# Choose "PAD" or "PCB"
DATA_TYPE = "PAD"

# Leave True for the first demonstration run.
USE_EXAMPLE_DATA = True

# Used when USE_EXAMPLE_DATA = False
INPUT_FILE = Path("pad_rate_curves_raw.csv")
OUTPUT_FILE = Path("pad_rate_curves_cleaned.csv")

# Cleaning settings
RESOLUTION_AXIS = "FORCE"       # Choose: "DISPLACEMENT" or "FORCE"
DISPLACEMENT_RESOLUTION_MM = 0.01
FORCE_RESOLUTION_N = 0.10
MEDIAN_FILTER_WINDOW = 5       # Must be an odd integer; use 1 to disable
BIN_VALUE_METHOD = "median"   # Choose: "median", "mean", or "max"
ENFORCE_NONDECREASING_FORCE = True
ADD_ZERO_POINT_IF_MISSING = True

# Output settings
SAVE_CLEANED_FILE = True

## Load raw data

Example mode intentionally contains repeated displacement readings, force noise, and a small force reversal so the effect of the cleaning process is visible. No example multiplier is used.

In [ ]:
def create_example_data(data_type):
    repeated_disp = [
        0.00, 0.00, 0.01, 0.01, 0.02, 0.02, 0.03, 0.04,
        0.04, 0.05, 0.06, 0.06, 0.08, 0.10, 0.10, 0.12,
        0.15, 0.18, 0.22, 0.26, 0.30, 0.35, 0.40,
    ]

    if data_type == "PAD":
        rows = []
        example_forces = {
            0.5: [
                0.00, 0.03, 0.06, 0.09, 0.13, 0.18, 0.25, 0.35,
                0.42, 0.55, 0.72, 0.68, 1.10, 1.65, 1.82, 2.40,
                3.50, 5.00, 7.20, 10.0, 13.5, 19.0, 26.0,
            ],
            5.0: [
                0.00, 0.05, 0.09, 0.14, 0.20, 0.28, 0.38, 0.52,
                0.63, 0.82, 1.05, 1.00, 1.55, 2.25, 2.45, 3.20,
                4.60, 6.40, 9.00, 12.3, 16.5, 23.0, 31.0,
            ],
        }
        for rate, forces in example_forces.items():
            for displacement, force in zip(repeated_disp, forces):
                rows.append({
                    "Rate_mm_min": rate,
                    "Disp_mm": displacement,
                    "Force_N": force,
                })
        return pd.DataFrame(rows)

    pcb_forces = [
        0.00, 0.15, 0.45, 0.55, 0.95, 1.10, 1.50, 1.95,
        2.10, 2.55, 3.00, 2.92, 4.05, 5.10, 5.20, 6.10,
        7.55, 9.00, 11.0, 13.0, 15.0, 17.5, 20.0,
    ]
    return pd.DataFrame({
        "Disp_mm": repeated_disp,
        "Force_N": pcb_forces,
    })


DATA_TYPE = DATA_TYPE.strip().upper()
RESOLUTION_AXIS = RESOLUTION_AXIS.strip().upper()
if DATA_TYPE not in {"PAD", "PCB"}:
    raise ValueError('DATA_TYPE must be either "PAD" or "PCB".')

if USE_EXAMPLE_DATA:
    raw_data = create_example_data(DATA_TYPE)
    print(f"Using illustrative {DATA_TYPE} data.")
else:
    raw_data = pd.read_csv(INPUT_FILE)
    print(f"Loaded raw data from: {INPUT_FILE.resolve()}")

raw_data.head(12)

## Validate and clean the loading curves

The median filter is applied in original acquisition order. Measurements are then grouped into resolution-sized bins on the selected channel. If displacement is selected, force is aggregated within each displacement bin. If force is selected, displacement is aggregated within each force bin. The default median is resistant to isolated spikes.

When enabled, the cumulative-maximum step prevents a noisy loading curve from decreasing with increasing compression. This is appropriate for the monotonic loading curves expected by the stack calculator, but it should not be used for unloading or hysteresis data.

In [ ]:
def validate_settings():
    if RESOLUTION_AXIS not in {"DISPLACEMENT", "FORCE"}:
        raise ValueError(
            'RESOLUTION_AXIS must be "DISPLACEMENT" or "FORCE".'
        )
    if DISPLACEMENT_RESOLUTION_MM <= 0:
        raise ValueError("DISPLACEMENT_RESOLUTION_MM must be greater than zero.")
    if FORCE_RESOLUTION_N <= 0:
        raise ValueError("FORCE_RESOLUTION_N must be greater than zero.")
    if (
        not isinstance(MEDIAN_FILTER_WINDOW, int)
        or MEDIAN_FILTER_WINDOW < 1
        or MEDIAN_FILTER_WINDOW % 2 == 0
    ):
        raise ValueError(
            "MEDIAN_FILTER_WINDOW must be a positive odd integer."
        )
    if BIN_VALUE_METHOD not in {"median", "mean", "max"}:
        raise ValueError(
            'BIN_VALUE_METHOD must be "median", "mean", or "max".'
        )


def prepare_raw_data(data, data_type):
    required = {"Disp_mm", "Force_N"}
    if data_type == "PAD":
        required.add("Rate_mm_min")

    missing = required.difference(data.columns)
    if missing:
        raise ValueError(
            "The input file is missing columns: "
            + ", ".join(sorted(missing))
        )

    prepared = data[list(required)].copy()
    prepared["_Original_Order"] = np.arange(len(prepared))

    for column in required:
        prepared[column] = pd.to_numeric(prepared[column], errors="coerce")

    invalid_count = int(prepared[list(required)].isna().any(axis=1).sum())
    if invalid_count:
        print(f"Removed {invalid_count} rows containing missing/non-numeric values.")
        prepared = prepared.dropna(subset=list(required)).copy()

    if (prepared[["Disp_mm", "Force_N"]] < 0).any().any():
        raise ValueError("Displacement and force values cannot be negative.")
    if data_type == "PAD" and (prepared["Rate_mm_min"] <= 0).any():
        raise ValueError("All pad compression rates must be greater than zero.")
    if len(prepared) < 2:
        raise ValueError("The input file needs at least two valid rows.")

    return prepared.sort_values("_Original_Order").reset_index(drop=True)


def clean_one_curve(curve):
    working = curve.sort_values("_Original_Order").copy()

    if RESOLUTION_AXIS == "DISPLACEMENT":
        working["Filtered_Value"] = (
            working["Force_N"]
            .rolling(
                window=MEDIAN_FILTER_WINDOW, center=True, min_periods=1
            )
            .median()
        )
        working["Resolution_Bin"] = (
            np.round(working["Disp_mm"] / DISPLACEMENT_RESOLUTION_MM)
            * DISPLACEMENT_RESOLUTION_MM
        )
        grouped = (
            working
            .groupby("Resolution_Bin", as_index=False)["Filtered_Value"]
            .agg(BIN_VALUE_METHOD)
            .rename(columns={
                "Resolution_Bin": "Disp_mm",
                "Filtered_Value": "Force_N",
            })
        )
    else:
        working["Filtered_Value"] = (
            working["Disp_mm"]
            .rolling(
                window=MEDIAN_FILTER_WINDOW, center=True, min_periods=1
            )
            .median()
        )
        working["Resolution_Bin"] = (
            np.round(working["Force_N"] / FORCE_RESOLUTION_N)
            * FORCE_RESOLUTION_N
        )
        grouped = (
            working
            .groupby("Resolution_Bin", as_index=False)["Filtered_Value"]
            .agg(BIN_VALUE_METHOD)
            .rename(columns={
                "Resolution_Bin": "Force_N",
                "Filtered_Value": "Disp_mm",
            })
        )

    grouped = grouped.sort_values("Disp_mm").reset_index(drop=True)
    grouped = (
        grouped
        .groupby("Disp_mm", as_index=False)["Force_N"]
        .median()
        .sort_values("Disp_mm")
        .reset_index(drop=True)
    )

    if ENFORCE_NONDECREASING_FORCE:
        grouped["Force_N"] = np.maximum.accumulate(
            grouped["Force_N"].to_numpy()
        )

    if ADD_ZERO_POINT_IF_MISSING and grouped.iloc[0]["Disp_mm"] > 0:
        grouped = pd.concat([
            pd.DataFrame({"Disp_mm": [0.0], "Force_N": [0.0]}),
            grouped,
        ], ignore_index=True)

    grouped["Disp_mm"] = grouped["Disp_mm"].round(6)
    grouped["Force_N"] = grouped["Force_N"].round(6)

    if len(grouped) < 2:
        raise ValueError(
            "Cleaning left fewer than two usable points. "
            "Use a smaller resolution for the selected channel."
        )
    return grouped


validate_settings()
prepared_raw_data = prepare_raw_data(raw_data, DATA_TYPE)

if DATA_TYPE == "PAD":
    cleaned_groups = []
    for rate, rate_curve in prepared_raw_data.groupby(
        "Rate_mm_min", sort=True
    ):
        cleaned_curve = clean_one_curve(rate_curve)
        cleaned_curve.insert(0, "Rate_mm_min", rate)
        cleaned_groups.append(cleaned_curve)
    cleaned_data = pd.concat(cleaned_groups, ignore_index=True)
    cleaned_data = cleaned_data[["Rate_mm_min", "Disp_mm", "Force_N"]]
else:
    cleaned_data = clean_one_curve(prepared_raw_data)
    cleaned_data = cleaned_data[["Disp_mm", "Force_N"]]

cleaned_data.head(12)

## Compare raw and cleaned curves

Confirm that the cleaned lines follow the central trend of the raw measurements without removing real features such as the initial rise or high-compression increase.

In [ ]:
def plot_comparison(raw, cleaned, data_type):
    fig, ax = plt.subplots(figsize=(11, 7))

    if data_type == "PAD":
        colors = plt.cm.viridis(
            np.linspace(0.1, 0.9, raw["Rate_mm_min"].nunique())
        )
        for color, rate in zip(colors, sorted(raw["Rate_mm_min"].unique())):
            raw_curve = raw[raw["Rate_mm_min"] == rate]
            clean_curve = cleaned[cleaned["Rate_mm_min"] == rate]
            ax.scatter(
                raw_curve["Disp_mm"],
                raw_curve["Force_N"],
                color=color,
                alpha=0.35,
                s=28,
                label=f"Raw: {rate:g} mm/min",
            )
            ax.plot(
                clean_curve["Disp_mm"],
                clean_curve["Force_N"],
                color=color,
                linewidth=2.5,
                marker="o",
                markersize=4,
                label=f"Cleaned: {rate:g} mm/min",
            )
    else:
        ax.scatter(
            raw["Disp_mm"], raw["Force_N"],
            color="tab:gray", alpha=0.45, s=32, label="Raw PCB",
        )
        ax.plot(
            cleaned["Disp_mm"], cleaned["Force_N"],
            color="tab:blue", linewidth=2.5, marker="o",
            markersize=4, label="Cleaned PCB",
        )

    ax.set_xlabel("Displacement (mm)")
    ax.set_ylabel("Force (N)")
    ax.set_title(f"{data_type} Data: Raw vs. Cleaned")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    plt.show()


plot_comparison(prepared_raw_data, cleaned_data, DATA_TYPE)

## Review and save the cleaned file

The saved CSV uses the exact columns expected by the rate-dependent stack calculator. The raw input file is never overwritten unless the same path is deliberately entered for both input and output; do not use the same path.

In [ ]:
if not USE_EXAMPLE_DATA:
    try:
        if INPUT_FILE.resolve() == OUTPUT_FILE.resolve():
            raise ValueError(
                "INPUT_FILE and OUTPUT_FILE must be different so the raw file "
                "is not overwritten."
            )
    except FileNotFoundError:
        pass

if DATA_TYPE == "PAD":
    summary_rows = []
    for rate in sorted(prepared_raw_data["Rate_mm_min"].unique()):
        summary_rows.append({
            "Curve": f"{rate:g} mm/min",
            "Raw_points": int((prepared_raw_data["Rate_mm_min"] == rate).sum()),
            "Cleaned_points": int((cleaned_data["Rate_mm_min"] == rate).sum()),
        })
    cleaning_summary = pd.DataFrame(summary_rows)
else:
    cleaning_summary = pd.DataFrame([{
        "Curve": "PCB",
        "Raw_points": len(prepared_raw_data),
        "Cleaned_points": len(cleaned_data),
    }])

print("Cleaning summary:")
print(cleaning_summary.to_string(index=False))

if SAVE_CLEANED_FILE:
    cleaned_data.to_csv(OUTPUT_FILE, index=False)
    print(f"\nCleaned CSV saved to: {OUTPUT_FILE.resolve()}")
else:
    print("\nThe cleaned file was not saved. Set SAVE_CLEANED_FILE = True to export it.")

cleaned_data

## Using the output

After reviewing the graph, use the cleaned file in the stack calculator:


```python
PAD_RATE_DATA_FILE = Path("pad_rate_curves_cleaned.csv")
PCB_DATA_FILE = Path("pcb_curve_cleaned.csv")
```

Run this cleaning notebook separately for the pad and PCB files. For PCB data that are intentionally modeled as perfectly linear, a linear regression may be preferable to filtering individual measurements.